In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile
zip_file = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("chat_data")
print("Files inside chat_data:", __import__("os").listdir("chat_data"))

Saving 13547360-hostel_bois (1).zip to 13547360-hostel_bois (1).zip
Files inside chat_data: ['hostel_bois.txt', 'GroupDNA_Minor_Project_Brief.pdf']


In [ ]:
import numpy as np
from datetime import datetime

CHAT_FILE = "chat_data/hostel_bois.txt"

In [23]:
def parse_chat(filename):
    messages = []
    current_msg = None
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "/" in line[:8]:
                try:
                    parts = line.split(" - ", 1)
                    if len(parts) < 2:
                        continue
                    timestamp = parts[0]
                    rest = parts[1]
                    if ": " in rest:
                        sender, text = rest.split(": ", 1)
                        if "<Media omitted>" in text or "deleted" in text.lower():
                            continue
                        current_msg = {
                            "datetime": timestamp,
                            "sender": sender.strip(),
                            "text": text.strip()
                        }
                        messages.append(current_msg)
                    else:
                        continue
                except:
                    continue
            else:
                if current_msg:
                    current_msg["text"] += " " + line
    return messages

In [24]:
def word_count(messages):
    words = {}
    stop_words = {
        "is","the","a","and","to","of","in",
        "on","for","with","at","by","an","be",
        "this","that","it","as","are","was"
    }
    for msg in messages:
        text = msg["text"].lower().split()
        for word in text:
            word = word.strip(".,!?")
            if word and word not in stop_words:
                words[word] = words.get(word, 0) + 1
    return words

In [25]:
import numpy as np
from datetime import datetime
def create_heatmap(messages):
    heatmap = np.zeros((7, 24), dtype=int)
    for msg in messages:
        dt = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M")
        day = dt.weekday()
        hour = dt.hour
        heatmap[day][hour] += 1
    return heatmap
def print_heatmap(heatmap):
    symbols = [" ", ".", "░", "▒", "▓", "█"]
    days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    print("\nACTIVITY HEATMAP (3-hour blocks)\n")
    hours = [f"{h:02d}" for h in range(0, 24, 3)]
    print("     " + "  ".join(hours))
    for i in range(7):
        row = heatmap[i]
        grouped = [sum(row[j:j+3]) for j in range(0, 24, 3)]
        max_val = max(grouped) if max(grouped) > 0 else 1
        line = ""
        for val in grouped:
            idx = int((val / max_val) * (len(symbols)-1))
            line += symbols[idx] + "  "
        print(f"{days[i]:<5} {line}")

In [26]:
from datetime import datetime
def response_times(messages):
    times = {}
    prev = None
    for msg in messages:
        curr_time = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M")
        sender = msg["sender"]
        if prev and prev["sender"] != sender:
            gap = (curr_time - prev["time"]).total_seconds()
            if sender not in times:
                times[sender] = []
            times[sender].append(gap)
        prev = {"sender": sender, "time": curr_time}
    avg = {}
    for user in times:
        avg[user] = sum(times[user]) / len(times[user])
    return avg
def silent_streak(messages):
    user_days = {}
    all_days = set()
    for msg in messages:
        dt = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M")
        user = msg["sender"]
        all_days.add(dt.date())
        if user not in user_days:
            user_days[user] = set()
        user_days[user].add(dt.date())
    total_days = len(all_days)
    streaks = {}
    for user in user_days:
        streaks[user] = total_days - len(user_days[user])
    return streaks

In [27]:
from datetime import datetime
def archetype(messages):
    user_stats = {}
    for msg in messages:
        user = msg["sender"]
        text = msg["text"]
        if user not in user_stats:
            user_stats[user] = {
                "count": 0,
                "words": 0,
                "night": 0,
                "caps": 0
            }
        user_stats[user]["count"] += 1
        user_stats[user]["words"] += len(text.split())
        dt = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M")
        if dt.hour >= 22 or dt.hour < 5:
            user_stats[user]["night"] += 1
        if text.isupper() and len(text) > 5:
            user_stats[user]["caps"] += 1
    result = {}
    for user, stats in user_stats.items():
        avg_words = stats["words"] / stats["count"]
        night_ratio = stats["night"] / stats["count"]
        if night_ratio > 0.6:
            result[user] = "NIGHT OWL"
        elif avg_words > 25:
            result[user] = "STORYTELLER"
        elif stats["caps"] > 30:
            result[user] = "DRAMA QUEEN"
        elif stats["count"] > 500:
            result[user] = "SPAMMER"
        elif stats["count"] < 50:
            result[user] = "GHOST"
        else:
            result[user] = "NORMAL"
    return result

In [32]:
messages = parse_chat(CHAT_FILE)
print("="*60)
print(" GROUPDNA ANALYSIS REPORT")
print("="*60)
print("\n OVERVIEW")
print("-"*60)
print("Total Messages:", len(messages))
users = set(msg["sender"] for msg in messages)
print("Total Participants:", len(users))
print("\n TOP WORDS")
print("-"*60)
words = word_count(messages)
sorted_words = sorted(words.items(), key=lambda x: x[1], reverse=True)
for word, count in sorted_words[:10]:
    bar = "█" * (count // 10 if count > 10 else 1)
    print(f"{word:<10} {bar} {count}")
print("\n ACTIVITY HEATMAP")
print("-"*60)
heatmap = create_heatmap(messages)
print_heatmap(heatmap)
print("\n RESPONSE TIMES (seconds)")
print("-"*60)
resp = response_times(messages)
for user, time in resp.items():
    print(f"{user:<15} {int(time)} sec")
print("\n SILENT STREAKS (days inactive)")
print("-"*60)
streaks = silent_streak(messages)
for user, days in streaks.items():
    print(f"{user:<15} {days} days")
print("\n USER ARCHETYPES")
print("-"*60)
roles = archetype(messages)
for user, role in roles.items():
    print(f"{user:<15} → {role}")
print("\n" + "="*60)
print(" END OF REPORT")
print("="*60)

 GROUPDNA ANALYSIS REPORT

 OVERVIEW
------------------------------------------------------------
Total Messages: 3127
Total Participants: 6

 TOP WORDS
------------------------------------------------------------
i          ███████████████████████████████████████████████████████████████████████████ 759
how        ████████████████████████████████ 321
guys       ███████████████████████████████ 318
you        ███████████████████████████████ 312
so         █████████████████████████████ 292
about      ███████████████████████████ 274
hai        ██████████████████████████ 268
am         ██████████████████████████ 260
today      █████████████████████████ 257
my         ██████████████████████ 223

 ACTIVITY HEATMAP
------------------------------------------------------------

ACTIVITY HEATMAP (3-hour blocks)

     00  03  06  09  12  15  18  21
Mon   .     .  .  ▒  ▒  █  ▒  
Tue   ░  ▒  .  ▒  ▒  ▒  █  ▒  
Wed   .  .  .  ▒  ▓  █  ▓  ▓  
Thu   .  .  .  ▒  ▓  ▒  █  ░  
Fri   ░  .  .  ▒  ▓  ▒  █  

In [35]:
print("\n INSIGHTS")
print("-"*60)
msg_count = {}
for m in messages:
    u = m["sender"]
    msg_count[u] = msg_count.get(u, 0) + 1
top_user = max(msg_count, key=msg_count.get)
print("1.", top_user, "is the most active user in the group.")
least_user = min(msg_count, key=msg_count.get)
print("2.", least_user, "is the least active user.")
top_word = max(words, key=words.get)
print("3. Most common word used is:", top_word)
from datetime import datetime
day_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
day_count = [0]*7
for m in messages:
    dt = datetime.strptime(m["datetime"], "%d/%m/%y, %H:%M")
    day_count[dt.weekday()] += 1
peak_day = day_names[day_count.index(max(day_count))]
print("4. Most active day is:", peak_day)
night_msgs = 0
for m in messages:
    dt = datetime.strptime(m["datetime"], "%d/%m/%y, %H:%M")
    if dt.hour >= 22 or dt.hour < 5:
        night_msgs += 1
print("5. Many messages are sent during night hours.")
fast_user = min(resp, key=resp.get)
print("6.", fast_user, "responds the fastest.")
silent_user = max(streaks, key=streaks.get)
print("7.", silent_user, "has the longest inactive streak.")
print("8. Group shows mixed personalities like NIGHT OWL, SPAMMER, etc.")


 INSIGHTS
------------------------------------------------------------
1. Rahul is the most active user in the group.
2. Vikas is the least active user.
3. Most common word used is: i
4. Most active day is: Mon
5. Many messages are sent during night hours.
6. Vikas responds the fastest.
7. Vikas has the longest inactive streak.
8. Group shows mixed personalities like NIGHT OWL, SPAMMER, etc.
